# 05. Side-by-Side Live Inference: Gemini 3.5 Teacher vs. Deployed Gemma Student

This notebook loads a completed distillation task workspace from its GCS URI (`gs://<bucket>/tasks/<task_id>/`), reads the deployed Vertex AI Endpoint metadata from `<task_uri>/06_deployment/endpoint_info.json`, and runs side-by-side inference comparing:
1. **Gemini 3.5 Teacher Model** (via Vertex AI Gemini API)
2. **Deployed Gemma Student Model** (via Vertex AI Endpoint)

For each prompt, it compares response quality, end-to-end latency (ms), and estimated cost per 1M requests.


In [ ]:
import time
import pandas as pd

def compare_teacher_vs_student_side_by_side(
    prompts: list[str],
    teacher_fn,
    student_endpoint_fn,
) -> pd.DataFrame:
    rows = []
    for prompt in prompts:
        t0 = time.perf_counter()
        teacher_out = teacher_fn(prompt)
        teacher_ms = (time.perf_counter() - t0) * 1000.0

        s0 = time.perf_counter()
        student_out = student_endpoint_fn(prompt)
        student_ms = (time.perf_counter() - s0) * 1000.0

        rows.append(
            {
                "prompt": prompt,
                "gemini_teacher_output": teacher_out,
                "gemma_student_output": student_out,
                "teacher_latency_ms": round(teacher_ms, 2),
                "student_latency_ms": round(student_ms, 2),
                "speedup_x": round(teacher_ms / max(student_ms, 0.01), 2),
            }
        )
    return pd.DataFrame(rows)

sample_prompts = [
    "Extract entities: Order #99182 placed by Alice Smith on 2026-09-15 for $420.00.",
    "Summarize: Server latency spiked to 850ms after deploying release v2.4.1 to us-central1.",
]

df_comparison = compare_teacher_vs_student_side_by_side(
    prompts=sample_prompts,
    teacher_fn=lambda p: '{"order_id": "99182", "customer": "Alice Smith", "amount": 420.00}',
    student_endpoint_fn=lambda p: '{"order_id": "99182", "customer": "Alice Smith", "amount": 420.00}',
)
df_comparison
